# AllPrep — exploration

Notebook d'exploration de l'étape 5 : jointures successives ventes, météo, proximité et récap ROD.

Objectif : visualiser chaque entrée, chaque jointure intermédiaire et remplir `../Output/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name != "AllPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
ROD_OUTPUT = PREPARE / "RodPrep" / "Output"
SALES_OUTPUT = PREPARE / "SalesPrep" / "Output"
METEO_OUTPUT = PREPARE / "MeteoPrep" / "Output"
PROX_OUTPUT = PREPARE / "ProximityPrep" / "Output"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Préparation `Input/` depuis les sorties des étapes précédentes

In [ ]:
sources = {
    "sales_joined": SALES_OUTPUT / "joined.parquet",
    "meteo_monthly": METEO_OUTPUT / "meteo_monthly.parquet",
    "proximity": PROX_OUTPUT / "proximity.parquet",
    "rod_hotel_lookup": ROD_OUTPUT / "hotel_lookup.parquet",
}

loaded = {}
for name, src in sources.items():
    if src.exists():
        df = pd.read_parquet(src)
        df.to_parquet(INPUT_DIR / f"{name}.parquet", index=False)
        loaded[name] = df
        print(f"{name:20s} → {df.shape}")
    else:
        print(f"{name:20s} → absent ({src.name}) — exécuter l'étape correspondante")

## 2. Lecture des entrées via `AllPrep._read`

In [ ]:
from all_prep.prep import AllPrep

all_prep = AllPrep(INPUT_DIR, OUTPUT_DIR)

sales = all_prep._read("sales_joined")
meteo = all_prep._read("meteo_monthly")
proximity = all_prep._read("proximity")
rod = all_prep._read("rod_hotel_lookup")

print("sales_joined", sales.shape)
sales.head() if not sales.empty else "vide"

In [ ]:
print("meteo_monthly", meteo.shape)
meteo.head() if not meteo.empty else "vide — optionnel si --skip-meteo"

In [ ]:
print("proximity", proximity.shape)
proximity.head() if not proximity.empty else "vide — optionnel si --skip-proximity"

In [ ]:
print("rod_hotel_lookup", rod.shape)
rod[["hotel_code", "hotel_name", "nom_hotel", "hotel_brand", "nb_chambres"]].head() if not rod.empty else "vide"

## 3. Jointure météo (clés `hotel_code`, `annee`, `mois`)

In [ ]:
result = sales.copy()
print(f"Base ventes : {result.shape[0]} lignes")

if not meteo.empty and not result.empty:
    keys = [k for k in ("hotel_code", "annee", "mois") if k in result.columns and k in meteo.columns]
    print("Clés météo :", keys)
    result = result.merge(meteo, on=keys, how="left", suffixes=("", "_meteo"))
    print(f"Après météo : {result.shape}")
    result.head()
else:
    print("Jointure météo ignorée (données absentes)")

## 4. Jointure proximité (clé `hotel_code`)

In [ ]:
if not proximity.empty and not result.empty and "hotel_code" in result.columns:
    before = result.shape[1]
    result = result.merge(proximity, on="hotel_code", how="left", suffixes=("", "_prox"))
    print(f"Après proximité : {result.shape[0]} lignes, +{result.shape[1] - before} colonnes")
    result[["hotel_code", "annee", "mois", "plage_distance_km", "commerce_fb_100m"]].head()
else:
    print("Jointure proximité ignorée")

## 5. Jointure récap ROD (colonnes non dupliquées)

In [ ]:
if not rod.empty and not result.empty and "hotel_code" in result.columns:
    rod_keys = [c for c in rod.columns if c not in result.columns or c == "hotel_code"]
    before = result.shape[1]
    result = result.merge(rod[rod_keys], on="hotel_code", how="left", suffixes=("", "_rod"))
    print(f"Après rod : {result.shape[0]} lignes, +{result.shape[1] - before} colonnes")
    recap_cols = [c for c in result.columns if str(c).startswith("d_recap_")][:6]
    show_cols = ["hotel_code", "nom_hotel", "annee", "mois"] + recap_cols
    result[[c for c in show_cols if c in result.columns]].head()
else:
    print("Jointure rod ignorée")

## 6. Nettoyage des noms de colonnes

In [ ]:
from prepare._shared.columns import sanitize_dataframe_columns

preview_cols = list(result.columns[:12])
sanitized = sanitize_dataframe_columns(list(result.columns))
renamed = pd.DataFrame({"avant": preview_cols, "apres": sanitized[:12]})
renamed

## 7. Persistance `Output/`

In [ ]:
dataset_full = all_prep.run()
print(f"dataset_full : {dataset_full.shape}")
dataset_full.head()

print("\nFichiers produits :")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(" ", path.name)